# DisasterLens BRIGHT training on Kaggle

This notebook runs M1–M2 against the attached official BRIGHT Kaggle Dataset. It does not create, download, or substitute data. Kaggle GPU execution is required and the run must report an NVIDIA Tesla T4.

In [ ]:
from pathlib import Path
import json
import os
import zipfile
import subprocess
import sys
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is unavailable. Set accelerator: gpu and rerun; no CPU fallback is allowed.')
gpu_name = torch.cuda.get_device_name(0)
print(f'[Kaggle] GPU: {gpu_name}', flush=True)
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'Tesla T4 required for this run, but Kaggle assigned: {gpu_name}')

candidates = [Path('/kaggle/working'), Path('/kaggle/input/disaster-lens')]
repo_dir = next((path for path in candidates if (path / 'pyproject.toml').is_file()), None)
if repo_dir is None:
    raise RuntimeError('Repository files were not uploaded to Kaggle working space; run this notebook through Kaggle Studio from the repository.')
repo_dir = repo_dir.resolve()
dataset_candidates = [Path('/kaggle/input/bright-dataset'), Path('/kaggle/input/bright')]
def find_bright_root(base):
    for root in [base, *base.rglob('*')]:
        if root.is_dir() and all((root / name).is_dir() for name in ('pre-event', 'post-event', 'target')):
            return root.resolve()
    return None
dataset_root = None
for candidate in dataset_candidates:
    if candidate.is_dir():
        dataset_root = find_bright_root(candidate)
        if dataset_root is not None:
            break
if dataset_root is None:
    archives = {}
    for candidate in dataset_candidates:
        if candidate.is_dir():
            for archive in candidate.rglob('*.zip'):
                for modality in ('pre-event', 'post-event', 'target'):
                    if modality in archive.stem.lower():
                        archives[modality] = archive
    if set(archives) == {'pre-event', 'post-event', 'target'}:
        extracted = Path('/kaggle/working/bright')
        extracted.mkdir(parents=True, exist_ok=True)
        for modality in ('pre-event', 'post-event', 'target'):
            print(f'[data] extracting official {archives[modality].name}', flush=True)
            with zipfile.ZipFile(archives[modality]) as archive:
                archive.extractall(extracted)
        dataset_root = find_bright_root(extracted)
if dataset_root is None:
    raise RuntimeError('Attached BRIGHT dataset must contain pre-event, post-event, and target directories.')
os.environ['DISASTERLENS_BRIGHT_ROOT'] = str(dataset_root)
print(f'[Kaggle] repository: {repo_dir}', flush=True)
print(f'[Kaggle] official BRIGHT root: {dataset_root}', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo_dir)], check=True)

In [ ]:
def run_step(title, command):
    print(f'\n{"=" * 72}\n{title}\n{"=" * 72}', flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
    print(f'[complete] {title} in {time.perf_counter() - started:.1f}s', flush=True)

run_step('[M1 1/3] Audit official BRIGHT data and write manifest', [sys.executable, '-u', 'scripts/inspect_bright.py', 'data=bright'])
run_step('[M1 2/3] Verify a real BRIGHT DataLoader batch', [sys.executable, '-u', 'scripts/verify_bright_loader.py', 'data=bright'])
events = sorted({json.loads(line)['event_id'] for line in (repo_dir / 'data/manifests/bright_manifest.jsonl').read_text().splitlines() if line.strip()})
if not events:
    raise RuntimeError('The official BRIGHT manifest contains no events.')
print(f'[M1 3/3] audited {len(events)} real events; first available event: {events[0]}', flush=True)

In [ ]:
# Optional override: set DISASTERLENS_TEST_EVENT before running this cell.
test_event = os.environ.get('DISASTERLENS_TEST_EVENT', events[0])
if test_event not in events:
    raise ValueError(f'Unknown TEST_EVENT {test_event!r}; choose one of the audited events, e.g. {events[:5]}')
print(f'[M2] using audited real event holdout: {test_event}', flush=True)
run_step('[M2 1/3] Create real event-holdout split', [sys.executable, '-u', 'scripts/make_splits.py', 'data=bright', f'split.test_events=[{test_event}]'])
run_step('[M2 2/3] Train the real-data baseline on the T4 (100 epochs)', [sys.executable, '-u', 'scripts/train.py', 'split_path=data/manifests/splits/event_holdout.json', 'overfit_tiles=8', 'trainer.epochs=100', 'trainer.crop_size=512'])
run_step('[M2 3/3] Evaluate the best checkpoint on the held-out event', [sys.executable, '-u', 'scripts/evaluate.py', 'checkpoint=outputs/checkpoints/early_fusion_unet/best.pt', 'split_path=data/manifests/splits/event_holdout.json', 'partition=test'])
print('[Kaggle] checkpoints, metrics, logs, reports, and manifests are under /kaggle/working/outputs and will be downloaded by Kaggle Studio.', flush=True)